In [2]:
%%capture
!pip install --upgrade transformers datasets accelerate evaluate jiwer soundfile librosa

In [3]:
import os
import re
import numpy as np
import pandas as pd
import torch
import librosa
import soundfile as sf
from dataclasses import dataclass
from typing import Any, Dict, List, Union

from datasets import Dataset, DatasetDict, Audio, load_dataset
from transformers import (
    WhisperFeatureExtractor,
    WhisperTokenizer,
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")

PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4


In [4]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────
MANIFEST_CSV   = "/kaggle/input/datasets/raghavmittal1234/whisper/manifest.csv"  # path to your manifest
AUDIO_BASE_DIR = "/kaggle/input/datasets/raghavmittal1234/whisper"         # folder containing .wav files
OUTPUT_DIR     = "/kaggle/working/whisper-small-hindi"
# ────────────────────────────────────────────────────────────────────────────

MODEL_ID       = "openai/whisper-small"
LANGUAGE       = "Hindi"
TASK           = "transcribe"
SAMPLING_RATE  = 16_000

# Training hyper-parameters (tuned for Kaggle T4 / 16 GB VRAM)
TRAIN_BATCH    = 16
EVAL_BATCH     = 8
GRAD_ACCUM     = 2       # effective batch = 32
WARMUP_STEPS   = 500
MAX_STEPS      = 4000
SAVE_STEPS     = 500
EVAL_STEPS     = 500
LEARNING_RATE  = 1e-5
MAX_INPUT_LEN  = 448     # Whisper decoder max token length

In [5]:
df = pd.read_csv(MANIFEST_CSV)


# ── Normalise column names ── adjust if your CSV uses different names
# Expected: 'path' (relative audio path) and 'transcript'
assert 'audio_path' in df.columns,       "CSV must have a 'path' column"
assert 'text' in df.columns, "CSV must have a 'transcript' column"
df['audio_path'] = df['audio_path'].str.replace("\\", "/", regex=False)
# Build absolute paths
df['abs_path'] = df['audio_path'].apply(
    lambda p: p if os.path.isabs(p) else os.path.join(AUDIO_BASE_DIR, p)
)
print(f"Total rows: {len(df)}")
print(df.head())
print("\nColumns:", df.columns.tolist())
# Drop rows where file doesn't exist or transcript is empty
before = len(df)
df = df[df['abs_path'].apply(os.path.exists)]
print(f"\nDropped {before - len(df)} rows (missing file or empty transcript)")
print(f"Usable rows: {len(df)}")

Total rows: 4469
                       audio_path  \
0  audio_segments/825780_0000.wav   
1  audio_segments/825780_0001.wav   
2  audio_segments/825780_0002.wav   
3  audio_segments/825780_0003.wav   
4  audio_segments/825780_0004.wav   

                                                text  duration  user_id  \
0  अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बह...     14.31   825780   
1  अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नह...     14.61   825780   
2  जंगल का सफर होता है जब हम रहने के लिए गए थे ना...     12.81   825780   
3  पहली बारी था क्योंकि चलना नहीं आता न वहाँ का ज...      8.10   825780   
4  हां तो फिर वहां जो दिन भर तो दिन में तो खोजने ...     14.13   825780   

   recording_id  segment_index  \
0        825780              0   
1        825780              1   
2        825780              2   
3        825780              3   
4        825780              4   

                                            abs_path  
0  /kaggle/input/datasets/raghavmittal1234/whispe

In [6]:
def normalise_hindi(text: str) -> str:
    """Light normalisation: strip extra whitespace, remove punctuation
    that Whisper wouldn't produce, normalise unicode."""
    if not isinstance(text, str):
        return ""
    text = text.strip()
    # Remove common punctuation (keep Devanagari danda '।' as Whisper may produce it)
    text = re.sub(r'[\"\',\-!?;:()\.\[\]{}]', '', text)
    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(normalise_hindi)

# Final drop after normalisation
df = df[df['text'] != '']
print(f"Final usable rows: {len(df)}")
print(df[['abs_path','text']].sample(5))

Final usable rows: 4469
                                               abs_path  \
1386  /kaggle/input/datasets/raghavmittal1234/whispe...   
1266  /kaggle/input/datasets/raghavmittal1234/whispe...   
239   /kaggle/input/datasets/raghavmittal1234/whispe...   
2553  /kaggle/input/datasets/raghavmittal1234/whispe...   
3965  /kaggle/input/datasets/raghavmittal1234/whispe...   

                                                   text  
1386  आदमी ज़्यादातर आदमी को क्या होता है पहले असफलत...  
1266  और खाने में हम ज़रूरी चीज़ें क्या भूल सकते हैं...  
239   इस फाइल में चला गया जो जो भी आप है उसमें डाल द...  
2553  नहीं जैसे हमारी मौसी आगरा रहती है तो फिर न हम ...  
3965                                  गाने सुनते हैं आप  


In [7]:
feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer         = WhisperTokenizer.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)
processor         = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)

print("Processor loaded. Tokenizer language:", tokenizer.language)

preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Processor loaded. Tokenizer language: hindi


In [9]:
from sklearn.model_selection import train_test_split
# Train/val split
train_df, val_df = train_test_split(df, test_size=0.05, random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

# PyTorch Dataset — no map, no cast_column
class WhisperDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_array, sr = sf.read(row["abs_path"])
        if sr != SAMPLING_RATE:
            audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=SAMPLING_RATE)
        if len(audio_array.shape) > 1:
            audio_array = audio_array.mean(axis=1)

        input_features = feature_extractor(
            audio_array, sampling_rate=SAMPLING_RATE
        ).input_features[0]

        labels = tokenizer(row["text"]).input_ids

        return {"input_features": input_features, "labels": labels}

train_dataset = WhisperDataset(train_df)
val_dataset   = WhisperDataset(val_df)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples  : {len(val_dataset)}")

Train: 4245 | Val: 224
Train samples: 4245
Val samples  : 224


In [10]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Pad input features
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad labels
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch   = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding token id with -100 so loss ignores padding
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Strip BOS token if present
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.generation_config.language = LANGUAGE
model.generation_config.task     = TASK
model.generation_config.forced_decoder_ids = None   # let processor handle it

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id
)
print("Model & data collator ready.")

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Model & data collator ready.


In [11]:
wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids   = pred.predictions
    label_ids  = pred.label_ids

    # Replace -100 padding back to pad token id for decoding
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str  = tokenizer.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": round(wer, 2)}

In [12]:
print("Loading FLEURS Hindi test set...")
fleurs_test = load_dataset(
    "google/fleurs", "hi_in",
    split="test",
    trust_remote_code=True
)
print(f"FLEURS test samples: {len(fleurs_test)}")
print(fleurs_test[0].keys())

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/fleurs' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading FLEURS Hindi test set...


README.md: 0.00B [00:00, ?B/s]

parquet-data/hi_in/train-00000-of-00001.(…):   0%|          | 0.00/1.53G [00:00<?, ?B/s]

parquet-data/hi_in/validation-00000-of-0(…):   0%|          | 0.00/162M [00:00<?, ?B/s]

parquet-data/hi_in/test-00000-of-00001.p(…):   0%|          | 0.00/306M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2120 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/239 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/418 [00:00<?, ? examples/s]

FLEURS test samples: 418
dict_keys(['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'])


In [13]:
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1

def evaluate_on_fleurs(model_id_or_path, label="Model"):
    """Evaluate Whisper on FLEURS Hindi test set using direct generate() — no pipeline."""

    # Load model + processor fresh for the given checkpoint
    eval_processor = WhisperProcessor.from_pretrained(
        model_id_or_path, language=LANGUAGE, task=TASK
    )
    eval_model = WhisperForConditionalGeneration.from_pretrained(model_id_or_path)
    eval_model.eval()
    eval_model = eval_model.to("cuda" if torch.cuda.is_available() else "cpu")

    forced_decoder_ids = eval_processor.get_decoder_prompt_ids(
        language=LANGUAGE, task=TASK
    )

    all_preds = []
    all_refs  = []

    for sample in fleurs_test:
        # Resample to 16kHz if needed
        audio_array = sample["audio"]["array"]
        sr          = sample["audio"]["sampling_rate"]
        if sr != SAMPLING_RATE:
            audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=SAMPLING_RATE)

        # Feature extraction
        inputs = eval_processor(
            audio_array,
            sampling_rate=SAMPLING_RATE,
            return_tensors="pt",
            return_attention_mask=True          # ← this line added
        )
 
        input_features = inputs.input_features.to(eval_model.device)
        attention_mask  = inputs.attention_mask.to(eval_model.device)
 
        # Pass attention_mask into generate
        with torch.no_grad():
            predicted_ids = eval_model.generate(
                input_features,
                attention_mask=attention_mask,  # ← and this line
                forced_decoder_ids=forced_decoder_ids,
            )

        pred = eval_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        ref  = sample["transcription"]

        all_preds.append(normalise_hindi(pred))
        all_refs.append(normalise_hindi(ref))

    wer = 100 * wer_metric.compute(predictions=all_preds, references=all_refs)
    print(f"\n{'='*45}")
    print(f"  {label}  |  FLEURS Hindi WER: {wer:.2f}%")
    print(f"{'='*45}")
    return round(wer, 2)
baseline_wer = evaluate_on_fleurs(MODEL_ID, label="Baseline whisper-small")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'tr


  Baseline whisper-small  |  FLEURS Hindi WER: 82.51%


In [16]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    # Batching
    per_device_train_batch_size=8,
    per_device_eval_batch_size=EVAL_BATCH,
    gradient_accumulation_steps=4,

    # Optimiser
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=500,
    fp16=True,      # mixed precision on GPU
    gradient_checkpointing=True,          # saves VRAM

    # Evaluation
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    # Generation (required for Seq2Seq eval)
    predict_with_generate=True,
    generation_max_length=MAX_INPUT_LEN,

    # Logging
    logging_steps=25,
    report_to="none",
    
    log_level="info",
    push_to_hub=False,

    # Misc
    dataloader_num_workers=2,
    save_total_limit=2,
)
print("Training args configured.")

Training args configured.


In [34]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [18]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)
from transformers import TrainerCallback

class StopAtStepCallback(TrainerCallback):
    def __init__(self, stop_step):
        self.stop_step = stop_step
    
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step >= self.stop_step:
            control.should_save = True
            control.should_training_stop = True
        return control

trainer.add_callback(StopAtStepCallback(stop_step=1500))
print("Starting fine-tuning...")
trainer.train()

# Save best checkpoint
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
from IPython.display import FileLink
import shutil
import os

# Zip the model
shutil.make_archive("/kaggle/working/whisper_hindi_finetuned", "zip", OUTPUT_DIR)

# Download link in notebook
FileLink("/kaggle/working/whisper_hindi_finetuned.zip")
print(f"\nModel saved to: {OUTPUT_DIR}")

max_steps is given, it will override any value given in num_train_epochs
***** Running training *****
  Num examples = 4,245
  Num Epochs = 8
  Instantaneous batch size per device = 8
  Training with DataParallel so batch size has been adjusted to: 16
  Total train batch size (w. parallel, distributed & accumulation) = 64
  Gradient Accumulation steps = 4
  Total optimization steps = 500
  Number of trainable parameters = 241,734,912


Starting fine-tuning...


Step,Training Loss,Validation Loss,Wer
100,2.682286,0.363491,39.790000
200,1.936427,0.324400,32.660000
300,1.277730,0.335737,32.340000
400,1.052078,0.341011,31.960000
500,0.846827,0.351563,32.030000



***** Running Evaluation *****
  Num examples = 224
  Batch size = 16
Increase max_length from 448 to 448 since input is conditioned on previous segment.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Increase max_length from 448 to 448 since input is conditioned on previous segment.
Increase max_length from 448 to 448 since input is conditioned on previous segment.
Increase max_length from 448 to 448 since input is conditioned on previous segment.
Increase max_length from 448 to 448 since input is conditioned on previous segment.
Increase max_length from 448 to 448 since input is conditioned on previous segment.
Increase max_length from 448 to 448 since input is conditioned on previous segment.
Increase max_length from 448 to 448 since input is conditioned on previous segment.
Increase max_length from 448

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in /kaggle/working/whisper-small-hindi/checkpoint-500/model.safetensors
Feature extractor saved in /kaggle/working/whisper-small-hindi/checkpoint-500/preprocessor_config.json
Deleting older checkpoint [/kaggle/working/whisper-small-hindi/checkpoint-300] due to args.save_total_limit


Training completed. Do not forget to share your model on huggingface.co/models =)


Loading best model from /kaggle/working/whisper-small-hindi/checkpoint-400 (score: 31.96).
There were missing keys in the checkpoint model loaded: ['proj_out.weight'].
Saving model checkpoint to /kaggle/working/whisper-small-hindi
Configuration saved in /kaggle/working/whisper-small-hindi/config.json
Configuration saved in /kaggle/working/whisper-small-hindi/generation_config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in /kaggle/working/whisper-small-hindi/model.safetensors
Feature extractor saved in /kaggle/working/whisper-small-hindi/preprocessor_config.json
tokenizer config file saved in /kaggle/working/whisper-small-hindi/tokenizer_config.json
processor saved in /kaggle/working/whisper-small-hindi/processor_config.json


KeyboardInterrupt: 

In [19]:
shutil.copy(
    "/kaggle/working/whisper_hindi_finetuned.zip",
    "/kaggle/working/whisper_hindi_finetuned_backup.zip"
)

'/kaggle/working/whisper_hindi_finetuned_backup.zip'

In [21]:
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="Hindi", task="transcribe"
)
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="Hindi", task="transcribe"
)
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

Configuration saved in /kaggle/working/whisper-small-hindi/config.json
Configuration saved in /kaggle/working/whisper-small-hindi/generation_config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in /kaggle/working/whisper-small-hindi/model.safetensors
tokenizer config file saved in /kaggle/working/whisper-small-hindi/tokenizer_config.json
processor saved in /kaggle/working/whisper-small-hindi/processor_config.json


['/kaggle/working/whisper-small-hindi/processor_config.json']

In [22]:
finetuned_wer = evaluate_on_fleurs(OUTPUT_DIR, label="Fine-tuned whisper-small")

loading configuration file /kaggle/working/whisper-small-hindi/processor_config.json
loading configuration file /kaggle/working/whisper-small-hindi/preprocessor_config.json
loading configuration file /kaggle/working/whisper-small-hindi/preprocessor_config.json
Feature extractor WhisperFeatureExtractor {
  "chunk_length": 30,
  "dither": 0.0,
  "feature_extractor_type": "WhisperFeatureExtractor",
  "feature_size": 80,
  "hop_length": 160,
  "n_fft": 400,
  "n_samples": 480000,
  "nb_max_frames": 3000,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": false,
  "sampling_rate": 16000
}

loading configuration file /kaggle/working/whisper-small-hindi/config.json
Model config WhisperConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "apply_spec_augment": false,
  "architectures": [
    "WhisperForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 50257,
  "classifier_proj_size": 256,
  "d_model": 768,
  "decoder_attenti

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

loading configuration file /kaggle/working/whisper-small-hindi/generation_config.json
Generate config GenerationConfig {
  "alignment_heads": [
    [
      5,
      3
    ],
    [
      5,
      9
    ],
    [
      8,
      0
    ],
    [
      8,
      4
    ],
    [
      8,
      7
    ],
    [
      8,
      8
    ],
    [
      9,
      0
    ],
    [
      9,
      7
    ],
    [
      9,
      9
    ],
    [
      10,
      5
    ]
  ],
  "assistant_confidence_threshold": 0.4,
  "assistant_lookbehind": 10,
  "begin_suppress_tokens": [
    220,
    50257
  ],
  "bos_token_id": 50257,
  "decoder_start_token_id": 50258,
  "diversity_penalty": 0.0,
  "do_sample": false,
  "early_stopping": false,
  "encoder_no_repeat_ngram_size": 0,
  "encoder_repetition_penalty": 1.0,
  "eos_token_id": 50257,
  "epsilon_cutoff": 0.0,
  "eta_cutoff": 0.0,
  "forced_decoder_ids": [
    [
      1,
      50276
    ],
    [
      2,
      50359
    ],
    [
      3,
      50363
    ]
  ],
  "is_multili


  Fine-tuned whisper-small  |  FLEURS Hindi WER: 40.23%


In [23]:
improvement = round(baseline_wer - finetuned_wer, 2)
rel_improve  = round((improvement / baseline_wer) * 100, 1) if baseline_wer > 0 else 0

results = pd.DataFrame({
    'Model'            : ['Baseline whisper-small (pretrained)',
                          'Fine-tuned whisper-small (ours)'],
    'Eval Set'         : ['FLEURS Hindi test', 'FLEURS Hindi test'],
    'Language'         : ['Hindi (hi)', 'Hindi (hi)'],
    'WER (%)'          : [baseline_wer, finetuned_wer],
    'Δ WER (pp)'       : ['—', f'-{improvement}'],
    'Relative Δ (%)'   : ['—', f'-{rel_improve}%'],
})

print("\n" + "="*70)
print("                   RESULTS SUMMARY")
print("="*70)
print(results.to_string(index=False))
print("="*70)
print(f"\nAbsolute WER reduction : {improvement} percentage points")
print(f"Relative WER reduction : {rel_improve}%")

# Also display as styled DataFrame in notebook
results.style.set_caption("WER Evaluation Results — FLEURS Hindi Test Set")\
             .highlight_min(subset=['WER (%)'], color='lightgreen')


                   RESULTS SUMMARY
                              Model          Eval Set   Language  WER (%) Δ WER (pp) Relative Δ (%)
Baseline whisper-small (pretrained) FLEURS Hindi test Hindi (hi)    82.51          —              —
    Fine-tuned whisper-small (ours) FLEURS Hindi test Hindi (hi)    40.23     -42.28         -51.2%

Absolute WER reduction : 42.28 percentage points
Relative WER reduction : 51.2%


,Model,Eval Set,Language,WER (%),Δ WER (pp),Relative Δ (%)
0,Baseline whisper-small (pretrained),FLEURS Hindi test,Hindi (hi),82.510000,—,—
1,Fine-tuned whisper-small (ours),FLEURS Hindi test,Hindi (hi),40.230000,-42.28,-51.2%


In [25]:
print("Sample predictions on FLEURS test (first 5 examples):\n")
print(f"{'#':<4} {'Reference':<60} {'Prediction':<60}")
print("-" * 125)

eval_processor = WhisperProcessor.from_pretrained(OUTPUT_DIR, language=LANGUAGE, task=TASK)
eval_model = WhisperForConditionalGeneration.from_pretrained(OUTPUT_DIR)
eval_model.eval()
eval_model = eval_model.to("cuda" if torch.cuda.is_available() else "cpu")
forced_decoder_ids = eval_processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)

for i in range(5):
    sample = fleurs_test[i]
    audio_array = sample["audio"]["array"]
    sr = sample["audio"]["sampling_rate"]
    if sr != SAMPLING_RATE:
        audio_array = librosa.resample(audio_array, orig_sr=sr, target_sr=SAMPLING_RATE)

    inputs = eval_processor(
        audio_array,
        sampling_rate=SAMPLING_RATE,
        return_tensors="pt",
        return_attention_mask=True
    )
    with torch.no_grad():
        predicted_ids = eval_model.generate(
            inputs.input_features.to(eval_model.device),
            attention_mask=inputs.attention_mask.to(eval_model.device),
            forced_decoder_ids=forced_decoder_ids,
        )
    pred = eval_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    ref  = normalise_hindi(sample["transcription"])
    pred = normalise_hindi(pred)
    print(f"{i+1:<4} {ref[:58]:<60} {pred[:58]:<60}")

loading configuration file /kaggle/working/whisper-small-hindi/processor_config.json
loading configuration file /kaggle/working/whisper-small-hindi/preprocessor_config.json
loading configuration file /kaggle/working/whisper-small-hindi/preprocessor_config.json
Feature extractor WhisperFeatureExtractor {
  "chunk_length": 30,
  "dither": 0.0,
  "feature_extractor_type": "WhisperFeatureExtractor",
  "feature_size": 80,
  "hop_length": 160,
  "n_fft": 400,
  "n_samples": 480000,
  "nb_max_frames": 3000,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": false,
  "sampling_rate": 16000
}

loading configuration file /kaggle/working/whisper-small-hindi/config.json
Model config WhisperConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "apply_spec_augment": false,
  "architectures": [
    "WhisperForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 50257,
  "classifier_proj_size": 256,
  "d_model": 768,
  "decoder_attenti

Sample predictions on FLEURS test (first 5 examples):

#    Reference                                                    Prediction                                                  
-----------------------------------------------------------------------------------------------------------------------------


Processor WhisperProcessor:
- feature_extractor: WhisperFeatureExtractor {
  "chunk_length": 30,
  "dither": 0.0,
  "feature_extractor_type": "WhisperFeatureExtractor",
  "feature_size": 80,
  "hop_length": 160,
  "n_fft": 400,
  "n_samples": 480000,
  "nb_max_frames": 3000,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": false,
  "sampling_rate": 16000
}

- tokenizer: WhisperTokenizer(name_or_path='/kaggle/working/whisper-small-hindi', vocab_size=50258, model_max_length=1024, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	50257: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50258: AddedToken("<|startoftranscript|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	50259: AddedToken("<|en|>", rstrip=False

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

loading configuration file /kaggle/working/whisper-small-hindi/generation_config.json
Generate config GenerationConfig {
  "alignment_heads": [
    [
      5,
      3
    ],
    [
      5,
      9
    ],
    [
      8,
      0
    ],
    [
      8,
      4
    ],
    [
      8,
      7
    ],
    [
      8,
      8
    ],
    [
      9,
      0
    ],
    [
      9,
      7
    ],
    [
      9,
      9
    ],
    [
      10,
      5
    ]
  ],
  "assistant_confidence_threshold": 0.4,
  "assistant_lookbehind": 10,
  "begin_suppress_tokens": [
    220,
    50257
  ],
  "bos_token_id": 50257,
  "decoder_start_token_id": 50258,
  "diversity_penalty": 0.0,
  "do_sample": false,
  "early_stopping": false,
  "encoder_no_repeat_ngram_size": 0,
  "encoder_repetition_penalty": 1.0,
  "eos_token_id": 50257,
  "epsilon_cutoff": 0.0,
  "eta_cutoff": 0.0,
  "forced_decoder_ids": [
    [
      1,
      50276
    ],
    [
      2,
      50359
    ],
    [
      3,
      50363
    ]
  ],
  "is_multili

...

2    ग्रीनलैंड को बहुत कम जगह बसाया गया था नॉर्स सगास में वे कह   ग्रीनलेंड को बहुत कम जगह बसाया गया था नौर्स सगास में वह कह  


Increase max_length from 448 to 448 since input is conditioned on previous segment.


3    ऐसी कोई वैश्विक परिभाषा नहीं है जिसके लिए निर्मित सामान एं   आईसी कोई वह स्पिक परिवाशा नहीं है जिसके लिए निर्मित समान ए  


Increase max_length from 448 to 448 since input is conditioned on previous segment.


4    टेलीविजन रिपोर्टों में प्लांट से निकलने वाला सफेद धुआं दिख   टेलिविजन रिपोर्टों में प्लांट से निकलने वाला सफे दुवां दिख  
5    इंटरनेट पर यह खोज शत्रुतापूर्ण पर्यावरण पाठ्यक्रम के लिए अ   इंटरनेट परिये खोज सत्रुतापुर्न परियावरंग पांठकरम के लिए अक  
